# Modul 6: Chaos

---

## Determinismus ohne Vorhersagbarkeit

> *"Vorhersagbarkeit: Führt der Flügelschlag eines Schmetterlings in Brasilien einen Tornado in Texas aus?"*
> — Edward Lorenz (1972)

Chaos ist eine der tiefgreifendsten Entdeckungen der Wissenschaft des 20. Jahrhunderts. Es zeigt, dass **Determinismus nicht Vorhersagbarkeit impliziert**.

Ein chaotisches System ist:
1. **Deterministisch**: Die Gleichungen sind exakt, kein Zufall
2. **Sensitiv auf Anfangsbedingungen**: Winzige Unterschiede wachsen exponentiell
3. **Aperiodisch**: Das System wiederholt sich nie exakt
4. **Beschränkt**: Die Trajektorie bleibt in einem endlichen Bereich

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({
    'figure.figsize': (12, 10),
    'figure.dpi': 100,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'lines.linewidth': 0.8,
})

%matplotlib inline

---

## 6.1 Das Lorenz-System: Die Geburt der Chaos-Theorie

1963 untersuchte der Meteorologe Edward Lorenz ein vereinfachtes Modell der atmosphärischen Konvektion:

$$\dot{x} = \sigma(y - x)$$
$$\dot{y} = x(\rho - z) - y$$
$$\dot{z} = xy - \beta z$$

**Parameter:**
- $\sigma$ = Prandtl-Zahl (Verhältnis von Viskosität zu Wärmeleitfähigkeit)
- $\rho$ = Rayleigh-Zahl (Antriebsstärke der Konvektion)
- $\beta$ = geometrischer Faktor

**Standardwerte:** $\sigma = 10$, $\rho = 28$, $\beta = 8/3$

In [ ]:
# Das Lorenz-System

def lorenz(state, t, sigma=10, rho=28, beta=8/3):
    x, y, z = state
    return [
        sigma * (y - x),
        x * (rho - z) - y,
        x * y - beta * z
    ]

# Der berühmte Lorenz-Attraktor
def plot_lorenz_attractor():
    fig = plt.figure(figsize=(16, 12))
    ax = fig.add_subplot(111, projection='3d')
    
    # Lange Integration
    t = np.linspace(0, 100, 20000)
    initial = [1, 1, 1]
    
    sol = odeint(lorenz, initial, t)
    
    # Farbe nach Zeit für visuellen Effekt
    colors = plt.cm.plasma(np.linspace(0, 1, len(t)))
    
    # Plot als Linie mit Farbverlauf (vereinfacht)
    ax.plot(sol[:, 0], sol[:, 1], sol[:, 2], 'b-', alpha=0.7, linewidth=0.4)
    
    # Die zwei "Flügel" markieren
    ax.scatter([np.sqrt(8/3 * 27)], [np.sqrt(8/3 * 27)], [27], 
              color='red', s=100, marker='o', label='Fixpunkt C⁺')
    ax.scatter([-np.sqrt(8/3 * 27)], [-np.sqrt(8/3 * 27)], [27],
              color='green', s=100, marker='o', label='Fixpunkt C⁻')
    ax.scatter([0], [0], [0], color='black', s=100, marker='o', label='Ursprung')
    
    ax.set_xlabel('X', fontsize=14)
    ax.set_ylabel('Y', fontsize=14)
    ax.set_zlabel('Z', fontsize=14)
    ax.set_title('Der Lorenz-Attraktor: "Der Schmetterling"', fontsize=18)
    ax.legend()
    
    return fig, ax, sol

fig, ax, sol = plot_lorenz_attractor()
plt.show()

### Der Lorenz-Attraktor: Beobachtungen

1. **Schmetterlingsform**: Zwei "Flügel" - die Trajektorie kreist um einen, springt zum anderen, zurück...

2. **Nie periodisch**: Die Trajektorie wiederholt sich nie exakt. Sie kommt beliebig nah an frühere Punkte, aber nie exakt.

3. **Beschränkt**: Obwohl aperiodisch, bleibt die Trajektorie in einem endlichen Bereich - dem **Attraktor**.

4. **Instabile Fixpunkte**: Der Ursprung und die beiden symmetrischen Punkte $C^\pm$ sind alle instabil - die Trajektorie wird immer weggestoßen.

---

## 6.2 Sensitive Abhängigkeit von Anfangsbedingungen

Das definierende Merkmal von Chaos: **Winzige Unterschiede in den Anfangsbedingungen führen zu völlig unterschiedlichen Trajektorien.**

In [ ]:
# Demonstration der sensitiven Abhängigkeit

def sensitive_dependence_demo():
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    t = np.linspace(0, 50, 10000)
    
    # Zwei fast identische Anfangsbedingungen
    ic1 = [1.0, 1.0, 1.0]
    ic2 = [1.0 + 1e-10, 1.0, 1.0]  # Unterschied: 0.0000000001 !
    
    sol1 = odeint(lorenz, ic1, t)
    sol2 = odeint(lorenz, ic2, t)
    
    # Zeitverläufe
    ax = axes[0, 0]
    ax.plot(t, sol1[:, 0], 'b-', label='IC 1', alpha=0.8)
    ax.plot(t, sol2[:, 0], 'r-', label='IC 2 (Δ = 10⁻¹⁰)', alpha=0.8)
    ax.set_xlabel('Zeit $t$')
    ax.set_ylabel('$x(t)$')
    ax.set_title('Zeitverlauf: Anfangs identisch...')
    ax.legend()
    ax.set_xlim(0, 50)
    
    # Zoom auf frühe Zeit
    ax = axes[0, 1]
    ax.plot(t, sol1[:, 0], 'b-', label='IC 1')
    ax.plot(t, sol2[:, 0], 'r--', label='IC 2')
    ax.set_xlabel('Zeit $t$')
    ax.set_ylabel('$x(t)$')
    ax.set_title('...dann divergierend')
    ax.legend()
    ax.set_xlim(20, 35)
    
    # Abstand zwischen den Trajektorien
    ax = axes[1, 0]
    distance = np.sqrt(np.sum((sol1 - sol2)**2, axis=1))
    ax.semilogy(t, distance, 'k-')
    ax.axhline(y=1e-10, color='r', linestyle='--', label='Anfangsabstand')
    ax.set_xlabel('Zeit $t$')
    ax.set_ylabel('Abstand $|\mathbf{x}_1 - \mathbf{x}_2|$')
    ax.set_title('Exponentielles Wachstum des Abstands')
    ax.legend()
    ax.set_xlim(0, 50)
    
    # 3D mit beiden Trajektorien
    ax = fig.add_subplot(2, 2, 4, projection='3d')
    ax.plot(sol1[:, 0], sol1[:, 1], sol1[:, 2], 'b-', alpha=0.6, linewidth=0.5)
    ax.plot(sol2[:, 0], sol2[:, 1], sol2[:, 2], 'r-', alpha=0.6, linewidth=0.5)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title('Beide Trajektorien im Phasenraum')
    
    plt.tight_layout()
    return fig

sensitive_dependence_demo()
plt.show()

### Die Bedeutung

Der Anfangsunterschied war $10^{-10}$ - ein Zehnmilliardstel!

Nach etwa 25 Zeiteinheiten sind die Trajektorien **völlig unkorreliert**.

In der Praxis bedeutet das:
- Jede Messung hat endliche Genauigkeit
- Der Fehler wächst exponentiell
- **Langfristige Vorhersage ist unmöglich**, selbst bei perfekter Kenntnis der Gleichungen

Das ist keine praktische Limitation - es ist eine **fundamentale Eigenschaft** des Systems.

---

## 6.3 Lyapunov-Exponenten: Chaos quantifizieren

Der **Lyapunov-Exponent** $\lambda$ misst die mittlere exponentielle Divergenzrate:

$$|\delta\mathbf{x}(t)| \approx |\delta\mathbf{x}(0)| \cdot e^{\lambda t}$$

**Interpretation:**
- $\lambda > 0$: **Chaos** (exponentielle Divergenz)
- $\lambda = 0$: Neutral (periodische Orbits, entlang des Flusses)
- $\lambda < 0$: Stabil (Konvergenz zum Attraktor)

Ein n-dimensionales System hat n Lyapunov-Exponenten, geordnet: $\lambda_1 \geq \lambda_2 \geq ... \geq \lambda_n$

In [ ]:
# Lyapunov-Exponent für 1D-Abbildungen

def lyapunov_exponent_1d(f, df, x0, n_iterations=10000, transient=1000):
    """
    Berechnet den Lyapunov-Exponenten für eine 1D-Abbildung.
    
    λ = lim_{n→∞} (1/n) Σ log|f'(x_i)|
    """
    x = x0
    
    # Transient überspringen
    for _ in range(transient):
        x = f(x)
    
    # Lyapunov-Summe
    lyap_sum = 0.0
    for _ in range(n_iterations):
        derivative = abs(df(x))
        if derivative > 0:
            lyap_sum += np.log(derivative)
        x = f(x)
    
    return lyap_sum / n_iterations

# Logistische Abbildung
def logistic(x, r):
    return r * x * (1 - x)

def logistic_derivative(x, r):
    return r * (1 - 2*x)

# Lyapunov-Exponent als Funktion von r
def lyapunov_vs_r():
    fig, ax = plt.subplots(figsize=(14, 6))
    
    r_values = np.linspace(2.5, 4.0, 1000)
    lyap_values = []
    
    for r in r_values:
        f = lambda x: logistic(x, r)
        df = lambda x: logistic_derivative(x, r)
        lyap = lyapunov_exponent_1d(f, df, 0.5)
        lyap_values.append(lyap)
    
    ax.plot(r_values, lyap_values, 'b-', linewidth=0.5)
    ax.axhline(y=0, color='r', linestyle='--', linewidth=2, label='λ = 0')
    ax.fill_between(r_values, lyap_values, 0, where=np.array(lyap_values) > 0,
                    alpha=0.3, color='red', label='Chaos (λ > 0)')
    ax.fill_between(r_values, lyap_values, 0, where=np.array(lyap_values) < 0,
                    alpha=0.3, color='blue', label='Ordnung (λ < 0)')
    
    ax.set_xlabel(r'Parameter $r$', fontsize=14)
    ax.set_ylabel(r'Lyapunov-Exponent $\lambda$', fontsize=14)
    ax.set_title('Lyapunov-Exponent der logistischen Abbildung', fontsize=16)
    ax.legend(loc='upper left')
    ax.set_xlim(2.5, 4.0)
    ax.set_ylim(-2, 1)
    
    # Wichtige Punkte
    ax.axvline(x=3.57, color='k', linestyle=':', alpha=0.5)
    ax.text(3.57, 0.8, 'Onset of Chaos', rotation=90, fontsize=10)
    
    return fig

lyapunov_vs_r()
plt.show()

### Beobachtungen

1. **$\lambda < 0$** für $r < 3.57$: Periodisches Verhalten (Konvergenz zum Attraktor)

2. **$\lambda > 0$** für $r > 3.57$ (meist): Chaos

3. **Negative Spitzen** im chaotischen Regime: Das sind die **Fenster der Ordnung** (periodische Fenster)

4. Bei $r = 4$: $\lambda = \ln 2$ (exakt berechenbar!)

### Für das Lorenz-System

Bei Standardparametern ($\sigma = 10$, $\rho = 28$, $\beta = 8/3$):

- $\lambda_1 \approx +0.91$ (Chaos-Richtung: Expansion)
- $\lambda_2 = 0$ (entlang des Flusses: neutral)
- $\lambda_3 \approx -14.6$ (starke Kontraktion)

**Summe:** $\lambda_1 + \lambda_2 + \lambda_3 \approx -13.7 < 0$

Das System ist **dissipativ** - Volumen schrumpft, aber trotzdem gibt es Chaos durch Expansion in einer Richtung und extremes Falten.

---

## 6.4 Der Mechanismus: Stretching und Folding

Wie kann ein System gleichzeitig:
- Volumen komprimieren (dissipativ)
- Trajektorien exponentiell trennen (chaotisch)
- Beschränkt bleiben

**Antwort:** Durch **Stretching (Dehnen)** und **Folding (Falten)**.

Stell dir vor, du knetest Teig:
1. **Strecken:** Der Teig wird dünner, benachbarte Punkte entfernen sich
2. **Falten:** Der Teig wird zurückgelegt, um im Behälter zu bleiben
3. **Wiederholen:** Nach vielen Zyklen sind ursprünglich benachbarte Punkte über den ganzen Teig verteilt

Das ist genau, was im Phasenraum passiert!

In [ ]:
# Demonstration mit der Baker's Map (vereinfachtes Modell)

def bakers_map(x, y):
    """Die Bäcker-Abbildung: Strecken und Falten."""
    if y < 0.5:
        return 2*x, y/2
    else:
        return 2*x - 1, (y + 1)/2

def demo_stretching_folding():
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    
    # Anfangspunkte: Ein kleines Quadrat
    n_points = 2000
    x = 0.2 + 0.1 * np.random.rand(n_points)
    y = 0.2 + 0.1 * np.random.rand(n_points)
    
    iterations = [0, 1, 3, 7]
    
    for iter_idx, n_iter in enumerate(iterations):
        # Kopie der Punkte
        x_temp, y_temp = x.copy(), y.copy()
        
        # Iterieren
        for _ in range(n_iter):
            x_new, y_new = [], []
            for xi, yi in zip(x_temp, y_temp):
                xn, yn = bakers_map(xi % 1, yi)  # mod 1 um im Einheitsquadrat zu bleiben
                x_new.append(xn % 1)
                y_new.append(yn)
            x_temp, y_temp = np.array(x_new), np.array(y_new)
        
        ax = axes[0, iter_idx]
        ax.scatter(x_temp, y_temp, s=0.5, c='blue', alpha=0.5)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect('equal')
        ax.set_title(f'Nach {n_iter} Iterationen')
        ax.set_xlabel('x')
        ax.set_ylabel('y')
    
    # Untere Reihe: Logistische Abbildung im chaotischen Regime
    r = 4.0
    x0_values = 0.5 + 0.01 * np.random.randn(n_points)
    
    for iter_idx, n_iter in enumerate([0, 5, 15, 50]):
        x_temp = x0_values.copy()
        for _ in range(n_iter):
            x_temp = r * x_temp * (1 - x_temp)
        
        ax = axes[1, iter_idx]
        ax.hist(x_temp, bins=50, density=True, color='blue', alpha=0.7)
        ax.set_xlim(0, 1)
        ax.set_xlabel('x')
        ax.set_ylabel('Dichte')
        ax.set_title(f'Verteilung nach {n_iter} Iterationen')
    
    plt.suptitle('Stretching und Folding: Wie Chaos Informationen mischt', fontsize=16, y=1.02)
    plt.tight_layout()
    return fig

demo_stretching_folding()
plt.show()

### Die Essenz

**Obere Reihe (Baker's Map):** Ein kleines Quadrat wird gestreckt und gefaltet. Nach wenigen Iterationen ist es über das ganze Einheitsquadrat verteilt - als Streifen.

**Untere Reihe (Logistische Abbildung):** Anfangs enge Verteilung wird nach wenigen Iterationen uniform über [0,1] verteilt.

**Information geht verloren:** Nach genug Iterationen weiß man nicht mehr, wo man angefangen hat. Die Anfangsbedingung ist "vergessen" - aber deterministisch!

---

## 6.5 Die logistische Abbildung: Chaos in 1D

Für kontinuierliche Flüsse braucht Chaos mindestens 3D (Poincaré-Bendixson).

Aber für **diskrete Abbildungen** reicht bereits 1D!

$$x_{n+1} = f(x_n)$$

Die logistische Abbildung $f(x) = rx(1-x)$ ist das einfachste chaotische System.

In [ ]:
# Vollständige Analyse der logistischen Abbildung

def logistic_analysis():
    fig = plt.figure(figsize=(18, 12))
    
    # 1. Bifurkationsdiagramm
    ax1 = fig.add_subplot(2, 2, 1)
    r_values = np.linspace(2.5, 4.0, 2000)
    
    x = 0.5 * np.ones(len(r_values))
    for _ in range(500):
        x = r_values * x * (1 - x)
    for _ in range(100):
        x = r_values * x * (1 - x)
        ax1.plot(r_values, x, 'k,', markersize=0.3)
    
    ax1.set_xlabel(r'$r$')
    ax1.set_ylabel(r'$x^*$')
    ax1.set_title('Bifurkationsdiagramm')
    
    # 2. Cobweb-Diagramm für r = 3.8
    ax2 = fig.add_subplot(2, 2, 2)
    r = 3.8
    x_line = np.linspace(0, 1, 500)
    ax2.plot(x_line, r * x_line * (1 - x_line), 'b-', linewidth=2, label=r'$f(x) = rx(1-x)$')
    ax2.plot(x_line, x_line, 'k--', linewidth=1, label=r'$y = x$')
    
    # Cobweb
    x = 0.1
    for _ in range(50):
        x_new = r * x * (1 - x)
        ax2.plot([x, x], [x, x_new], 'r-', alpha=0.5, linewidth=0.5)
        ax2.plot([x, x_new], [x_new, x_new], 'r-', alpha=0.5, linewidth=0.5)
        x = x_new
    
    ax2.set_xlabel(r'$x_n$')
    ax2.set_ylabel(r'$x_{n+1}$')
    ax2.set_title(f'Cobweb-Diagramm für r = {r}')
    ax2.legend()
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    
    # 3. Zeitreihe für r = 4
    ax3 = fig.add_subplot(2, 2, 3)
    r = 4.0
    x = 0.1
    trajectory = [x]
    for _ in range(100):
        x = r * x * (1 - x)
        trajectory.append(x)
    
    ax3.plot(trajectory, 'b-', linewidth=0.8)
    ax3.set_xlabel('Iteration $n$')
    ax3.set_ylabel(r'$x_n$')
    ax3.set_title(f'Zeitreihe für r = {r} (voll chaotisch)')
    
    # 4. Sensitivität
    ax4 = fig.add_subplot(2, 2, 4)
    r = 4.0
    x1, x2 = 0.5, 0.5 + 1e-14
    diff = [abs(x1 - x2)]
    
    for _ in range(50):
        x1 = r * x1 * (1 - x1)
        x2 = r * x2 * (1 - x2)
        diff.append(abs(x1 - x2))
    
    ax4.semilogy(diff, 'b-', linewidth=1.5)
    ax4.axhline(y=1e-14, color='r', linestyle='--', label='Anfangsabstand')
    ax4.set_xlabel('Iteration $n$')
    ax4.set_ylabel('Abstand (log)')
    ax4.set_title(r'Sensitivität: $\Delta x_0 = 10^{-14}$')
    ax4.legend()
    
    plt.tight_layout()
    return fig

logistic_analysis()
plt.show()

---

## 6.6 Chaos und Determinismus: Die philosophische Dimension

Chaos stellt fundamentale Fragen:

### Determinismus vs. Vorhersagbarkeit

Laplace (1814): *"Eine Intelligenz, die alle Kräfte der Natur und die gegenseitige Lage aller Wesen kennt, würde in einer Formel Vergangenheit und Zukunft des Universums umfassen."*

Chaos zeigt: **Selbst mit perfekten Gleichungen ist Vorhersage limitiert.**

Nicht weil die Welt nicht deterministisch wäre, sondern weil:
- Anfangsbedingungen nie perfekt bekannt sind
- Exponentielles Wachstum von Fehlern jede Präzision zerstört

### Der Vorhersagehorizont

Wenn $\lambda$ der größte Lyapunov-Exponent ist und $\epsilon$ die Anfangsgenauigkeit:

$$t_{\text{Vorhersage}} \sim \frac{1}{\lambda} \ln\left(\frac{\Delta}{\epsilon}\right)$$

wobei $\Delta$ die relevante Systemgröße ist.

**Beispiel Wetter:** $\lambda \approx 1/5$ Tag⁻¹ → Verdopplung des Fehlers alle 5 Tage → praktische Grenze ~2 Wochen

---

## 6.7 Zusammenfassung

### Was ist Chaos?

1. **Deterministisch** aber **unvorhersagbar** (langfristig)
2. **Sensitiv** auf Anfangsbedingungen (positiver Lyapunov-Exponent)
3. **Aperiodisch** aber **beschränkt**
4. Entsteht durch **Stretching + Folding**

### Schlüsselkonzepte

- **Lyapunov-Exponent:** $\lambda > 0$ definiert Chaos
- **Seltsamer Attraktor:** Beschränkte Menge mit fraktaler Struktur
- **Periodenverdopplung:** Universeller Weg ins Chaos
- **Universalität:** Feigenbaum-Konstanten gelten für viele Systeme

### Was kommt

Im nächsten Modul: **Seltsame Attraktoren und Fraktale Dimension** - Die geometrische Struktur des Chaos. Warum hat der Lorenz-Attraktor Dimension ~2.06? Was bedeutet "fraktale Dimension"?

---

## Übungen

### Übung 6.1
Berechne den Lyapunov-Exponenten für die Zeltabbildung $f(x) = 1 - |2x - 1|$ für verschiedene Startwerte. Ist er konstant?

### Übung 6.2
Implementiere eine Funktion, die den größten Lyapunov-Exponenten des Lorenz-Systems numerisch schätzt.

### Übung 6.3 (Reflexion)
Wenn ein System deterministisch ist, warum können wir das Wetter nicht langfristig vorhersagen? Erkläre in eigenen Worten.

In [ ]:
# Raum für deine Lösungen

